In [3]:


DATA_DIR        = r"./course-planner-rag"
DB_DIR          = r"./chroma_db23"
COLLECTION_NAME = "course_planner"
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 150


import hashlib
from pathlib import Path

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

print("✅ All imports successful")




def chunk_text(text: str) -> list:
    chunks, start = [], 0
    text = text.strip()
    while start < len(text):
        chunk = text[start : start + CHUNK_SIZE].strip()
        if chunk:
            chunks.append(chunk)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def stable_id(file_path: str, chunk_index: int) -> str:
    return hashlib.md5(f"{file_path}::chunk_{chunk_index}".encode()).hexdigest()


def load_txt_files(data_dir: Path) -> list:
    documents = []
    seen_paths = set()

    for subfolder in ["course", "policies", "programs"]:
        folder = data_dir / subfolder
        if not folder.exists():
            print(f"  [WARN] Folder not found, skipping: {folder}")
            continue

        txt_files = sorted(folder.glob("*.txt"))
        unique_files = []
        for filepath in txt_files:
            resolved = filepath.resolve()
            if resolved in seen_paths:
                print(f"    [SKIP] Duplicate: {filepath.name}")
                continue
            seen_paths.add(resolved)
            unique_files.append(filepath)

        print(f"  {subfolder:10s} → {len(unique_files)} file(s) found")

        for filepath in unique_files:
            try:
                text = filepath.read_text(encoding="utf-8", errors="replace").strip()
                if not text:
                    print(f"    [SKIP] Empty: {filepath.name}")
                    continue
                documents.append({
                    "text":     text,
                    "source":   str(filepath.relative_to(data_dir)),
                    "category": subfolder,
                    "filename": filepath.name,
                })
            except Exception as e:
                print(f"    [ERROR] {filepath}: {e}")

    return documents

print("✅ Helper functions defined")


# ── CELL 5: Load txt files ────────────────────────────────────────────────────

print(f"\n[1/4] Loading .txt files from: {DATA_DIR}\n")
documents = load_txt_files(Path(DATA_DIR))

if not documents:
    raise FileNotFoundError(f"No .txt files found under {DATA_DIR}.")

print(f"\n  Total documents loaded: {len(documents)}")


# ── CELL 6: Chunk documents ───────────────────────────────────────────────────

print(f"\n[2/4] Chunking (size={CHUNK_SIZE} chars, overlap={CHUNK_OVERLAP} chars) ...")

all_chunks, all_ids, all_metadata = [], [], []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(stable_id(doc["source"], i))
        all_metadata.append({
            "source":       doc["source"],
            "category":     doc["category"],
            "filename":     doc["filename"],
            "chunk_index":  i,
            "total_chunks": len(chunks),
        })

print(f"  Total chunks created: {len(all_chunks)}")


# ── CELL 7: Load model and generate embeddings ────────────────────────────────

print("\n[3/4] Loading BAAI/bge-small-en-v1.5 (~133 MB download on first run) ...")

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("  Generating embeddings ...")

# BGE models expect a query prefix for retrieval tasks
passages = [f"Represent this sentence: {c}" for c in all_chunks]

embeddings = model.encode(
    passages,
    batch_size=32,
    normalize_embeddings=True,   # cosine similarity
    show_progress_bar=True,
).tolist()

print(f"  ✅ Done — embedding dim: {len(embeddings[0])}")   # 384


# ── CELL 8: Store in ChromaDB ─────────────────────────────────────────────────

print(f"\n[4/4] Storing in ChromaDB at: {DB_DIR}")

db_path = Path(DB_DIR)
db_path.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(
    path=str(db_path),
    settings=Settings(anonymized_telemetry=False),
)

existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    print(f"  Deleting existing collection '{COLLECTION_NAME}' ...")
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

BATCH = 200
for start in range(0, len(all_chunks), BATCH):
    end = start + BATCH
    collection.add(
        ids        = all_ids[start:end],
        embeddings = embeddings[start:end],
        documents  = all_chunks[start:end],
        metadatas  = all_metadata[start:end],
    )
    print(f"  Stored chunks {start+1}–{min(end, len(all_chunks))} / {len(all_chunks)}")

print(f"\n✅ Indexing complete!")
print(f"   Collection : {COLLECTION_NAME}")
print(f"   Chunks     : {collection.count()}")
print(f"   Dimensions : 384  (BAAI/bge-small-en-v1.5)")
print(f"   DB path    : {db_path.resolve()}")


# ── CELL 9: Sanity check ──────────────────────────────────────────────────────

TEST_QUERY = "course graduation requirements"
print(f'\n─── Sanity check: "{TEST_QUERY}" ───')

test_vec = model.encode(
    [f"Represent this sentence: {TEST_QUERY}"],
    normalize_embeddings=True,
).tolist()

results = collection.query(
    query_embeddings=test_vec,
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    score = 1 - dist
    print(f"\n  Result {i+1}  |  score={score:.4f}  |  [{meta['category']}] {meta['filename']}  (chunk {meta['chunk_index']})")
    print(f"  {doc[:180]}{'...' if len(doc) > 180 else ''}")

print("\n🎉 Pipeline complete — ChromaDB is ready for RAG queries!")

✅ All imports successful
✅ Helper functions defined

[1/4] Loading .txt files from: ./course-planner-rag

  course     → 20 file(s) found
  policies   → 4 file(s) found
  programs   → 2 file(s) found

  Total documents loaded: 26

[2/4] Chunking (size=800 chars, overlap=150 chars) ...
  Total chunks created: 36

[3/4] Loading BAAI/bge-small-en-v1.5 (~133 MB download on first run) ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Generating embeddings ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  ✅ Done — embedding dim: 384

[4/4] Storing in ChromaDB at: ./chroma_db23
  Stored chunks 1–36 / 36

✅ Indexing complete!
   Collection : course_planner
   Chunks     : 36
   Dimensions : 384  (BAAI/bge-small-en-v1.5)
   DB path    : D:\anaconda3\Scripts\chroma_db23

─── Sanity check: "course graduation requirements" ───

  Result 1  |  score=0.6877  |  [policies] undergraduate_academic_standards.txt  (chunk 1)
  -----------------------------------------------

Evaluation Process:

If a student does not meet the minimum academic
standards, the Committee on Academic Performance
reviews the s...

  Result 2  |  score=0.6870  |  [programs] ai_decision_program.txt  (chunk 1)
  ligence

6.036 Introduction to Machine Learning

6.3900 Introduction to Deep Learning

-------------------------------------------------

Systems Requirement:

6.1800 Computer Syst...

  Result 3  |  score=0.6848  |  [course] 6.100L Introduction to Computer Science and Programming.txt  (chunk 0)
  Course Code: 6.100

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   Course-Planner RAG — Query Script                         ║
# ║   Model : BAAI/bge-small-en-v1.5                            ║
# ╚══════════════════════════════════════════════════════════════╝

# ── CELL 1: Configuration ─────────────────────────────────────────────────────

DB_DIR          = r"./chroma_db23"
COLLECTION_NAME = "course_planner"

QUERY    = "What are the graduation requirements?"
CATEGORY = None    # "course" | "policy" | "programs" | None (search all)
TOP_K    = 3


# ── CELL 2: Imports ───────────────────────────────────────────────────────────

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

print("✅ Imports successful")


# ── CELL 3: Load model + collection ──────────────────────────────────────────

client = chromadb.PersistentClient(
    path=DB_DIR,
    settings=Settings(anonymized_telemetry=False),
)
collection = client.get_collection(COLLECTION_NAME)
print(f"✅ Collection '{COLLECTION_NAME}' loaded — {collection.count()} chunks")

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("✅ Model loaded")


# ── CELL 4: Run query ─────────────────────────────────────────────────────────

vec = model.encode(
    [f"Represent this sentence: {QUERY}"],
    normalize_embeddings=True,
).tolist()

where   = {"category": CATEGORY} if CATEGORY else None
results = collection.query(
    query_embeddings=vec,
    n_results=TOP_K,
    where=where,
    include=["documents", "metadatas", "distances"],
)

print(f"\nQuery  : {QUERY}")
if CATEGORY:
    print(f"Filter : category = {CATEGORY}")
print(f"Top-{TOP_K} results:\n{'─'*60}")

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    score = 1 - dist
    print(f"\n[{i+1}] Score   : {score:.4f}")
    print(f"     File    : {meta['filename']}  ({meta['category']})")
    print(f"     Content : {doc[:300]}{'...' if len(doc) > 300 else ''}")

print()

✅ Imports successful
✅ Collection 'course_planner' loaded — 36 chunks
✅ Model loaded

Query  : What are the graduation requirements?
Top-3 results:
────────────────────────────────────────────────────────────

[1] Score   : 0.7024
     File    : undergraduate_academic_standards.txt  (policies)
     Content : -----------------------------------------------

Evaluation Process:

If a student does not meet the minimum academic
standards, the Committee on Academic Performance
reviews the student's academic record.

The review considers:
- grades received
- number of units taken
- type of subjects
- progress...

[2] Score   : 0.6511
     File    : ai_decision_program.txt  (programs)
     Content : ligence

6.036 Introduction to Machine Learning

6.3900 Introduction to Deep Learning

-------------------------------------------------

Systems Requirement:

6.1800 Computer Systems Engineering
or
6.1810 Operating System Engineering

-------------------------------------------------

Electives:


In [7]:
from typing import Dict, Any
from typing import TypedDict,Any, Annotated, Sequence,Optional
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, SystemMessage,AnyMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END,START
from langgraph.prebuilt import ToolNode
import json
from langchain_core.messages import RemoveMessage
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image,display
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool
from langgraph.graph import MessagesState
from pydantic import BaseModel, Field
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langchain_core.prompts import PromptTemplate
from typing import Literal
import pandas as pd
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
import matplotlib.pyplot as plt
import seaborn as sns
from langgraph.prebuilt import create_react_agent
from typing import TypedDict, Annotated, Any, Optional, Literal
from langgraph.graph import StateGraph
from langchain_core.messages import AnyMessage
import operator
import os
from langfuse import observe
from langchain.tools import tool
from langgraph.graph.message import add_messages 
from typing import TypedDict, Annotated, Optional, Literal, Dict, List, Any
from langchain.tools import tool
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings


In [3]:
class AgentState(TypedDict):
    # ── Student Input ──────────────────────────────
    student_query: str
    completed_courses: List[str]
    grades: Optional[dict]           # optional per doc
    target_program: Optional[str]
    target_term: Optional[str]       # Fall/Spring
    max_credits: Optional[int]
    catalog_year: Optional[str]
    transfer_credits: Optional[List[str]]

    # ── Intake Agent ───────────────────────────────
    missing_fields: List[str]
    clarifying_questions: List[str]  # 1-5 as per doc
    needs_clarification: bool

    # ── Retriever Agent ────────────────────────────
    retrieved_chunks: List[dict]     # {text, url, section, chunk_id}

    # ── Planner Agent ──────────────────────────────
    suggested_courses: List[dict]    # {course, justification, prereq_satisfied}
    risks_assumptions: List[str]     # e.g. "availability not in docs"

    # ── Prerequisite Reasoning (mandatory in doc) ──
    decision: Optional[str]          # Eligible / Not eligible / Need more info
    evidence: List[str]              # citations supporting decision
    next_step: Optional[str]         # what to take/do next

    # ── Verifier Agent ─────────────────────────────
    verification_passed: bool
    unsupported_claims: List[str]
    citation_coverage: Optional[float]  # % responses with citations (eval metric)

    # ── Safe Abstention (mandatory in doc) ─────────
    is_not_in_catalog: bool
    suggested_next_action: Optional[str]  # "check advisor/dept page/schedule"

    # ── Final Output (doc format) ──────────────────
    answer: str                      # Answer / Plan
    why: str                         # requirements/prereqs satisfied
    citations: List[str]             # URL + section heading or chunk_id
    assumptions: List[str]           # Not in catalog

In [67]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-3-haiku",  # Best for tool calling
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)
llm.invoke("Explain Langraph in one word")

AIMessage(content='Concise.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 15, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.25e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.25e-05, 'upstream_inference_prompt_cost': 3.75e-06, 'upstream_inference_completions_cost': 8.75e-06}}, 'model_provider': 'openai', 'model_name': 'anthropic/claude-3-haiku', 'system_fingerprint': None, 'id': 'gen-1774825153-FIYz57bF6CDnZCkGVa93', 'service_tier': 'standard', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3bd2-9e60-7210-bc4e-fa6e4e29b419-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 7, 'total_token

In [69]:
llm = ChatOpenAI(
    model="llama-3.1-8b-instant",
    api_key="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",
    base_url="https://api.groq.com/openai/v1",
    temperature=0
)
llm.invoke("Who is hitler")

AIMessage(content="Adolf Hitler (April 20, 1889 - April 30, 1945) was an Austrian-born German politician and leader of the Nazi Party (Nationalsozialistische Deutsche Arbeiterpartei, or NSDAP). He is widely regarded as one of the most infamous and reviled individuals in world history.\n\n**Early Life and Rise to Power**\n\nHitler was born in Braunau am Inn, Austria, to Alois Hitler and Klara Pölzl. He grew up in a poor family and was rejected from the Academy of Fine Arts Vienna twice. He served in the German army during World War I and was awarded the Iron Cross for bravery.\n\nAfter the war, Hitler joined the German Workers' Party (DAP), which later became the Nazi Party. He quickly rose to prominence, becoming the party's leader in 1921. Hitler's charisma, oratory skills, and promise of a return to German greatness resonated with many Germans who were struggling economically and feeling humiliated by the Treaty of Versailles.\n\n**Nazi Ideology and Policies**\n\nHitler's Nazi Party 

In [62]:
# ══════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════
import re
from typing import TypedDict, List, Optional
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# ══════════════════════════════════════════════════════
# LLM SETUP
# ══════════════════════════════════════════════════════
llm = ChatOpenAI(
    model="anthropic/claude-3-haiku",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

# ══════════════════════════════════════════════════════
# CHROMADB SETUP
# ══════════════════════════════════════════════════════
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

client = chromadb.PersistentClient(
    path="./chroma_db23",
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_collection("course_planner")

# ══════════════════════════════════════════════════════
# STATE
# ══════════════════════════════════════════════════════
class AgentState(TypedDict):
    # Student Input
    student_query:      str
    completed_courses:  List[str]
    grades:             Optional[dict]
    target_program:     Optional[str]
    target_term:        Optional[str]
    max_credits:        Optional[int]
    catalog_year:       Optional[str]
    transfer_credits:   Optional[List[str]]
    # Intake Agent
    missing_fields:         List[str]
    clarifying_questions:   List[str]
    needs_clarification:    bool
    # Retriever Agent
    retrieved_chunks:   List[dict]
    # Planner Agent
    suggested_courses:  List[dict]
    risks_assumptions:  List[str]
    # Prerequisite Reasoning
    decision:   Optional[str]
    evidence:   List[str]
    next_step:  Optional[str]
    # Verifier Agent
    verification_passed:    bool
    unsupported_claims:     List[str]
    citation_coverage:      Optional[float]
    retry_count:            int          # ── prevent infinite loop
    # Safe Abstention
    is_not_in_catalog:      bool
    suggested_next_action:  Optional[str]
    # Final Output
    answer:      str
    why:         str
    citations:   List[str]
    assumptions: List[str]

# ══════════════════════════════════════════════════════
# RETRIEVER TOOL
# ══════════════════════════════════════════════════════
@tool
def retrieve_catalog_chunks(query: str) -> list:
    """Retrieves relevant course catalog chunks with citations from ChromaDB."""

    query_vec = embed_model.encode(
        [f"Represent this sentence: {query}"],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_vec,
        n_results=15,
        include=["documents", "metadatas", "distances"]
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text":     doc,
            "source":   meta.get("source", ""),
            "category": meta.get("category", ""),
            "filename": meta.get("filename", ""),
            "chunk_id": meta.get("chunk_index", ""),
            "score":    round(1 - dist, 4),
            "citation": f"{meta.get('filename', '')} | chunk_{meta.get('chunk_index', '')}"
        })

    return chunks

# ══════════════════════════════════════════════════════
# HELPER
# ══════════════════════════════════════════════════════
def _extract_section(text: str, section: str) -> str:
    try:
        start = text.index(section) + len(section) + 1
        remaining = text[start:]
        next_section = re.search(r'\n[A-Z][^\n]+:\n', remaining)
        end = start + next_section.start() if next_section else len(text)
        return text[start:end].strip()
    except ValueError:
        return ""

# ══════════════════════════════════════════════════════
# AGENT 1 — INTAKE NODE
# ══════════════════════════════════════════════════════
def intake_node(state: AgentState) -> AgentState:
    missing = []

    if not state.get("target_program"):
        missing.append("target_program")
    if not state.get("completed_courses"):
        missing.append("completed_courses")
    if not state.get("target_term"):
        missing.append("target_term")
    if not state.get("max_credits"):
        missing.append("max_credits")
    if not state.get("catalog_year"):
        missing.append("catalog_year")

    # grades and transfer_credits are OPTIONAL per PDF

    if missing:
        prompt = f"""
You are a student academic advisor assistant.

Student asked: {state['student_query']}
Missing information: {missing}

Rules:
- Ask MAX 5 clarifying questions
- One question per missing field
- Be friendly and concise
- Do NOT guess or assume any information
"""
        response = llm.invoke(prompt)
        return {
            **state,
            "missing_fields":       missing,
            "clarifying_questions": [response.content],
            "needs_clarification":  True
        }

    return {
        **state,
        "missing_fields":       [],
        "clarifying_questions": [],
        "needs_clarification":  False
    }

# ══════════════════════════════════════════════════════
# AGENT 2 — RETRIEVER NODE
# ══════════════════════════════════════════════════════
def retriever_node(state: AgentState) -> AgentState:
    completed  = state.get('completed_courses', [])
    seen_ids   = set()
    all_chunks = []

    # ── Step 1: Embedding search by course codes ──
    for course in completed:
        queries = [
            f"prerequisite {course}",
            f"{course} OR",
            f"{course} AND",
            course,
        ]
        for query in queries:
            chunks = retrieve_catalog_chunks.invoke(query)
            for chunk in chunks:
                if chunk['chunk_id'] not in seen_ids:
                    seen_ids.add(chunk['chunk_id'])
                    all_chunks.append(chunk)

    # ── Step 2: Filter embedding results ──
    filtered_chunks = []
    for chunk in all_chunks:
        text_lower   = chunk['text'].lower()
        mentions     = any(c.lower() in text_lower for c in completed)
        is_completed = any(chunk['filename'].startswith(c) for c in completed)
        if mentions and not is_completed:
            filtered_chunks.append(chunk)

    # ── Step 3: Full collection scan ──
    # Catches courses embedding model missed
    # No hardcoding — dynamically checks completed courses
    all_results = collection.get(include=["documents", "metadatas"])

    for doc, meta in zip(all_results["documents"], all_results["metadatas"]):
        filename  = meta.get("filename", "")
        chunk_id  = str(meta.get("chunk_index", ""))
        unique_id = f"{filename}_{chunk_id}"

        if unique_id in seen_ids:
            continue

        text_lower   = doc.lower()
        mentions     = any(c.lower() in text_lower for c in completed)
        is_completed = any(filename.startswith(c) for c in completed)

        if mentions and not is_completed:
            seen_ids.add(unique_id)
            filtered_chunks.append({
                "text":     doc,
                "source":   meta.get("source", ""),
                "category": meta.get("category", ""),
                "filename": filename,
                "chunk_id": chunk_id,
                "score":    0.0,
                "citation": f"{filename} | chunk_{chunk_id}"
            })

    print(f"\nDEBUG Retriever: Found {len(filtered_chunks)} chunks")
    for c in filtered_chunks:
        print(f"  → {c['filename']} | score: {c['score']}")

    return {
        **state,
        "retrieved_chunks": filtered_chunks
    }
# ══════════════════════════════════════════════════════
# AGENT 3 — PLANNER NODE
# ══════════════════════════════════════════════════════
def planner_node(state: AgentState) -> AgentState:

    context = "\n\n".join([
        f"[{c['filename']} | chunk_{c['chunk_id']}]\n{c['text']}"
        for c in state['retrieved_chunks']
    ])

    prompt = f"""
You are a strict academic course planner.

Student Profile:
- Completed Courses: {state.get('completed_courses')}
- Grades: {state.get('grades')}
- Target Program: {state.get('target_program')}
- Target Term: {state.get('target_term')}
- Max Credits: {state.get('max_credits')}
- Catalog Year: {state.get('catalog_year')}

Catalog Context:
{context}

IMPORTANT — WHAT NEEDS CITATIONS:
- Citations are ONLY needed for catalog facts (prerequisites, course descriptions)
- Student grades and completed courses come from student profile — NO citation needed for these
- Do NOT try to cite student input data

PREREQUISITE LOGIC — ABSOLUTE RULES:

AND RULE (CRITICAL):
- "A AND B" = student MUST have BOTH A and B
- Student has ONLY A but NOT B → NOT ELIGIBLE — NO EXCEPTIONS
- Do NOT suggest this course under any circumstances
- Move to "Not Eligible Courses" with reason

OR RULE:
- "A OR B" = student needs ONLY ONE
- Student has A → ELIGIBLE even without B

EXAMPLES:
- Prerequisites: 6.1020 AND 6.1910
  Student has: [6.1020]
  Result: NOT ELIGIBLE — missing 6.1910

- Prerequisites: 6.1020 OR 6.1910
  Student has: [6.1020]
  Result: ELIGIBLE — has 6.1020

ELIGIBILITY CHECK (do for EVERY course in context):
Step 1 - Read prerequisites from catalog context
Step 2 - AND or OR?
Step 3 - AND → ALL in {state.get('completed_courses')}? Any missing → NOT ELIGIBLE
Step 4 - OR  → ONE in {state.get('completed_courses')}? One present → ELIGIBLE
Step 5 - Eligible → Answer/Plan | Not eligible → Not Eligible Courses
Step 6 - Unclear → Not in catalog

Output EXACTLY in this format:

Answer / Plan:
<only courses student IS eligible for with citation>

Not Eligible Courses:
- <course>: Missing prerequisites → <exactly which ones>

Why (requirements/prereqs satisfied):
<per course AND/OR logic explicitly>

Evidence:
- <catalog citations only>

Citations:
- <filename | chunk_id> for each catalog fact

Clarifying Questions:
- None

Assumptions / Not in catalog:
- <anything not found in catalog context>

DECISION: Eligible / Not Eligible / Need More Info

RISKS/ASSUMPTIONS:
- <risks>

NEXT STEP:
- <what student should do>
"""

    response = llm.invoke(prompt)
    text = response.content

    return {
        **state,
        "answer":                text,
        "decision":              _extract_section(text, "DECISION:"),
        "risks_assumptions":     _extract_section(text, "RISKS/ASSUMPTIONS:").split("\n"),
        "next_step":             _extract_section(text, "NEXT STEP:"),
        "why":                   _extract_section(text, "Why (requirements/prereqs satisfied):"),
        "citations":             _extract_section(text, "Citations:").split("\n"),
        "evidence":              _extract_section(text, "Evidence:").split("\n"),
        "assumptions":           _extract_section(text, "Assumptions / Not in catalog:").split("\n"),
        "is_not_in_catalog":     "Not in catalog" in text,
        "suggested_next_action": _extract_section(text, "NEXT STEP:"),
    }

# ══════════════════════════════════════════════════════
# AGENT 4 — VERIFIER NODE
# ══════════════════════════════════════════════════════
def verifier_node(state: AgentState) -> AgentState:

    valid_citations = [
        f"{c['filename']} | chunk_{c['chunk_id']}"
        for c in state['retrieved_chunks']
    ]

    prompt = f"""
You are a strict academic policy verifier/auditor.

Verify this course plan:
{state['answer']}

Valid catalog citations:
{valid_citations}

Student profile (this is INPUT data — does NOT need citations):
- Completed Courses: {state.get('completed_courses')}
- Grades: {state.get('grades')}

VERIFICATION RULES:
1. Catalog facts (prerequisites, course descriptions) MUST have citations
2. Student profile data (completed courses, grades) does NOT need citations
3. AND prerequisites — ALL must be present, if ANY missing → course must be in Not Eligible
4. OR prerequisites — ONE is enough → course can be in Answer/Plan
5. Citations must be from valid catalog citations list above

PASS CONDITIONS (ALL must be true):
- Every catalog fact has a valid citation
- AND logic applied correctly — missing AND prereq = Not Eligible
- OR logic applied correctly — one OR prereq = Eligible
- No catalog facts stated without citations

Output EXACTLY in this format:

VERIFICATION: PASSED / FAILED

UNSUPPORTED CLAIMS:
- <only catalog claims without valid citations>

CITATION COVERAGE:
- <cited catalog claims> / <total catalog claims>

FEEDBACK:
- <specific fixes needed if FAILED>
"""

    response = llm.invoke(prompt)
    text = response.content

    # Calculate citation coverage
    retrieved_count = len(state['retrieved_chunks'])
    cited_count = sum(
        1 for c in valid_citations
        if c in state['answer']
    )
    coverage = round(cited_count / retrieved_count, 2) if retrieved_count > 0 else 0.0
    verification_passed = "VERIFICATION: PASSED" in text

    unsupported = _extract_section(text, "UNSUPPORTED CLAIMS:")
    unsupported_list = [
        u.strip("- ").strip()
        for u in unsupported.split("\n")
        if u.strip()
    ]

    # Increment retry count
    retry_count = state.get('retry_count', 0) + 1

    return {
        **state,
        "verification_passed": verification_passed,
        "unsupported_claims":  unsupported_list,
        "citation_coverage":   coverage,
        "retry_count":         retry_count,
        "answer": state['answer'] if verification_passed else (
            state['answer'] + f"\n\n⚠️ VERIFICATION FAILED:\n{text}"
        )
    }

# ══════════════════════════════════════════════════════
# CONDITIONAL EDGES
# ══════════════════════════════════════════════════════
def should_clarify(state: AgentState) -> str:
    if state['needs_clarification']:
        return "end"
    return "retriever"

def should_rewrite(state: AgentState) -> str:
    # MAX 2 retries to prevent infinite loop
    if not state['verification_passed'] and state.get('retry_count', 0) < 2:
        return "planner"
    return "end"

# ══════════════════════════════════════════════════════
# BUILD GRAPH
# ══════════════════════════════════════════════════════
workflow = StateGraph(AgentState)

workflow.add_node("intake",    intake_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("planner",   planner_node)
workflow.add_node("verifier",  verifier_node)

workflow.set_entry_point("intake")

workflow.add_conditional_edges(
    "intake",
    should_clarify,
    {
        "end":       END,
        "retriever": "retriever"
    }
)

workflow.add_edge("retriever", "planner")
workflow.add_edge("planner",   "verifier")

workflow.add_conditional_edges(
    "verifier",
    should_rewrite,
    {
        "planner": "planner",
        "end":     END
    }
)

app = workflow.compile()

# ══════════════════════════════════════════════════════
# TEST RUN
# ══════════════════════════════════════════════════════
for step in app.stream({
    "student_query":        "What courses can I take next semester?",
    "completed_courses":    ["6.1020"],
    "grades":               {"6.1020": "A"},
    "target_program":       "Computer Science",
    "target_term":          "Fall",
    "max_credits":          48,
    "catalog_year":         "2024",
    "transfer_credits":     [],
    "missing_fields":       [],
    "clarifying_questions": [],
    "needs_clarification":  False,
    "retrieved_chunks":     [],
    "suggested_courses":    [],
    "risks_assumptions":    [],
    "decision":             "",
    "evidence":             [],
    "next_step":            "",
    "verification_passed":  False,
    "unsupported_claims":   [],
    "citation_coverage":    0.0,
    "retry_count":          0,
    "is_not_in_catalog":    False,
    "suggested_next_action":"",
    "answer":               "",
    "why":                  "",
    "citations":            [],
    "assumptions":          []
}):
    print("─── Node Output ───")
    for node, output in step.items():
        print(f"\n NODE: {node}")
        if 'answer' in output and output['answer']:
            print(output['answer'])
        elif 'clarifying_questions' in output and output['clarifying_questions']:
            print(output['clarifying_questions'])
        elif 'retrieved_chunks' in output:
            print(f"Retrieved {len(output['retrieved_chunks'])} chunks")
    print()

─── Node Output ───

 NODE: intake
Retrieved 0 chunks


DEBUG Retriever: Found 9 chunks
  → 6.1100 Computer Language Engineering.txt | score: 0.7643
  → 6.1040 Software Design.txt | score: 0.0
  → 6.1060 Software Performance Engineering.txt | score: 0.0
  → 6.1100 Computer Language Engineering.txt | score: 0.0
  → 6.1120 Dynamic Computer Language Engineering.txt | score: 0.0
  → 6.5120 Formal Reasoning About Programs.txt | score: 0.0
  → ai_decision_program.txt | score: 0.0
  → cs_engineering_program.txt | score: 0.0
  → cs_engineering_program.txt | score: 0.0
─── Node Output ───

 NODE: retriever
Retrieved 9 chunks

─── Node Output ───

 NODE: planner
Answer / Plan:
- 6.1040 Software Design [6.1040 Software Design.txt | chunk_0]: Student has completed 6.1020, which satisfies the prerequisite.
- 6.5120 Formal Reasoning About Programs [6.5120 Formal Reasoning About Programs.txt | chunk_0]: Student has completed 6.1020, which satisfies one of the prerequisites. The other prerequisite, 6.

In [63]:
# Run this separately to debug
test_queries = [
    "6.1020 OR 6.1910",
    "6.1120",
    "Dynamic Computer Language",
    "prerequisite 6.1020 or"
]

for q in test_queries:
    query_vec = embed_model.encode(
        [f"Represent this sentence: {q}"],
        normalize_embeddings=True
    ).tolist()
    
    results = collection.query(
        query_embeddings=query_vec,
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )
    
    print(f"\nQuery: '{q}'")
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"  → {meta['filename']} | score: {round(1-dist, 4)}")


Query: '6.1020 OR 6.1910'
  → 6.1000 Introduction to Programming and Computer Science.txt | score: 0.7157
  → 6.1100 Computer Language Engineering.txt | score: 0.7002
  → 6.1010 Fundamentals of Programming.txt | score: 0.6943

Query: '6.1120'
  → 6.1100 Computer Language Engineering.txt | score: 0.6635
  → 6.1000 Introduction to Programming and Computer Science.txt | score: 0.6633
  → ai_decision_program.txt | score: 0.6519

Query: 'Dynamic Computer Language'
  → 6.1120 Dynamic Computer Language Engineering.txt | score: 0.7856
  → 6.1100 Computer Language Engineering.txt | score: 0.6849
  → cs_engineering_program.txt | score: 0.6581

Query: 'prerequisite 6.1020 or'
  → 6.1100 Computer Language Engineering.txt | score: 0.7558
  → 6.1010 Fundamentals of Programming.txt | score: 0.7553
  → 6.1020 Software Construction.txt | score: 0.7379


In [64]:
# ══════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════
import re
from typing import TypedDict, List, Optional
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# ══════════════════════════════════════════════════════
# LLM SETUP
# ══════════════════════════════════════════════════════
llm = ChatOpenAI(
    model="anthropic/claude-3-haiku",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

# ══════════════════════════════════════════════════════
# CHROMADB SETUP
# ══════════════════════════════════════════════════════
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

client = chromadb.PersistentClient(
    path="./chroma_db23",
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_collection("course_planner")

# ══════════════════════════════════════════════════════
# STATE
# ══════════════════════════════════════════════════════
class AgentState(TypedDict):
    # Student Input
    student_query:      str
    completed_courses:  List[str]
    grades:             Optional[dict]
    target_program:     Optional[str]
    target_term:        Optional[str]
    max_credits:        Optional[int]
    catalog_year:       Optional[str]
    transfer_credits:   Optional[List[str]]
    # Intake Agent
    missing_fields:         List[str]
    clarifying_questions:   List[str]
    needs_clarification:    bool
    # Retriever Agent
    retrieved_chunks:   List[dict]
    # Planner Agent
    suggested_courses:  List[dict]
    risks_assumptions:  List[str]
    # Prerequisite Reasoning
    decision:   Optional[str]
    evidence:   List[str]
    next_step:  Optional[str]
    # Verifier Agent
    verification_passed:    bool
    unsupported_claims:     List[str]
    citation_coverage:      Optional[float]
    retry_count:            int
    # Safe Abstention
    is_not_in_catalog:      bool
    suggested_next_action:  Optional[str]
    # Final Output
    answer:      str
    why:         str
    citations:   List[str]
    assumptions: List[str]

# ══════════════════════════════════════════════════════
# RETRIEVER TOOL
# ══════════════════════════════════════════════════════
@tool
def retrieve_catalog_chunks(query: str) -> list:
    """Retrieves relevant course catalog chunks with citations from ChromaDB."""

    query_vec = embed_model.encode(
        [f"Represent this sentence: {query}"],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_vec,
        n_results=15,
        include=["documents", "metadatas", "distances"]
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text":     doc,
            "source":   meta.get("source", ""),
            "category": meta.get("category", ""),
            "filename": meta.get("filename", ""),
            "chunk_id": meta.get("chunk_index", ""),
            "score":    round(1 - dist, 4),
            "citation": f"{meta.get('filename', '')} | chunk_{meta.get('chunk_index', '')}"
        })

    return chunks

# ══════════════════════════════════════════════════════
# HELPER
# ══════════════════════════════════════════════════════
def _extract_section(text: str, section: str) -> str:
    try:
        start = text.index(section) + len(section) + 1
        remaining = text[start:]
        next_section = re.search(r'\n[A-Z][^\n]+:\n', remaining)
        end = start + next_section.start() if next_section else len(text)
        return text[start:end].strip()
    except ValueError:
        return ""

# ══════════════════════════════════════════════════════
# AGENT 1 — INTAKE NODE
# ══════════════════════════════════════════════════════
def intake_node(state: AgentState) -> AgentState:
    missing = []

    if not state.get("target_program"):
        missing.append("target_program")
    if not state.get("completed_courses"):
        missing.append("completed_courses")
    if not state.get("target_term"):
        missing.append("target_term")
    if not state.get("max_credits"):
        missing.append("max_credits")
    if not state.get("catalog_year"):
        missing.append("catalog_year")

    if missing:
        prompt = f"""
You are a student academic advisor assistant.

Student asked: {state['student_query']}
Missing information: {missing}

Rules:
- Ask MAX 5 clarifying questions
- One question per missing field
- Be friendly and concise
- Do NOT guess or assume any information
"""
        response = llm.invoke(prompt)
        return {
            **state,
            "missing_fields":       missing,
            "clarifying_questions": [response.content],
            "needs_clarification":  True
        }

    return {
        **state,
        "missing_fields":       [],
        "clarifying_questions": [],
        "needs_clarification":  False
    }

# ══════════════════════════════════════════════════════
# AGENT 2 — RETRIEVER NODE
# ══════════════════════════════════════════════════════
def retriever_node(state: AgentState) -> AgentState:
    completed  = state.get('completed_courses', [])
    seen_ids   = set()
    all_chunks = []

    # Step 1: Embedding search
    for course in completed:
        queries = [
            f"prerequisite {course}",
            f"{course} OR",
            f"{course} AND",
            course,
        ]
        for query in queries:
            chunks = retrieve_catalog_chunks.invoke(query)
            for chunk in chunks:
                uid = f"{chunk['filename']}_{chunk['chunk_id']}"
                if uid not in seen_ids:
                    seen_ids.add(uid)
                    all_chunks.append(chunk)

    # Step 2: Filter embedding results
    filtered_chunks = []
    for chunk in all_chunks:
        text_lower   = chunk['text'].lower()
        mentions     = any(c.lower() in text_lower for c in completed)
        is_completed = any(chunk['filename'].startswith(c) for c in completed)
        if mentions and not is_completed:
            filtered_chunks.append(chunk)

    # Step 3: Full collection scan
    # Catches anything embedding missed — fully dynamic
    all_results = collection.get(include=["documents", "metadatas"])

    for doc, meta in zip(all_results["documents"], all_results["metadatas"]):
        filename  = meta.get("filename", "")
        chunk_id  = str(meta.get("chunk_index", ""))
        uid       = f"{filename}_{chunk_id}"

        if uid in seen_ids:
            continue

        text_lower   = doc.lower()
        mentions     = any(c.lower() in text_lower for c in completed)
        is_completed = any(filename.startswith(c) for c in completed)

        if mentions and not is_completed:
            seen_ids.add(uid)
            filtered_chunks.append({
                "text":     doc,
                "source":   meta.get("source", ""),
                "category": meta.get("category", ""),
                "filename": filename,
                "chunk_id": chunk_id,
                "score":    0.0,
                "citation": f"{filename} | chunk_{chunk_id}"
            })

    print(f"\nDEBUG Retriever: Found {len(filtered_chunks)} chunks")
    for c in filtered_chunks:
        print(f"  → {c['filename']} | score: {c['score']}")

    return {
        **state,
        "retrieved_chunks": filtered_chunks
    }

# ══════════════════════════════════════════════════════
# AGENT 3 — PLANNER NODE
# ══════════════════════════════════════════════════════
def planner_node(state: AgentState) -> AgentState:

    context = "\n\n".join([
        f"[{c['filename']} | chunk_{c['chunk_id']}]\n{c['text']}"
        for c in state['retrieved_chunks']
    ])

    prompt = f"""
You are a strict academic course planner.

Student Profile:
- Completed Courses: {state.get('completed_courses')}
- Grades: {state.get('grades')}
- Target Program: {state.get('target_program')}
- Target Term: {state.get('target_term')}
- Max Credits: {state.get('max_credits')}
- Catalog Year: {state.get('catalog_year')}

Catalog Context:
{context}

IMPORTANT — CITATIONS:
- Citations needed ONLY for catalog facts (prerequisites, descriptions)
- Student grades and completed courses are INPUT data — NO citation needed

PREREQUISITE LOGIC — FOLLOW EXACTLY:

AND RULE (ABSOLUTE — NO EXCEPTIONS):
- "A and B" = student MUST have BOTH A and B
- Student missing ANY one → NOT ELIGIBLE
- Examples:
  * "6.1020 and 6.1200" → need BOTH
  * Student has [6.1020] only → NOT ELIGIBLE (missing 6.1200)
  * Student has [6.1020, 6.1200] → ELIGIBLE

OR RULE (ABSOLUTE — NO EXCEPTIONS):
- "A or B" = student needs ONLY ONE of A or B
- Student has ANY ONE → ELIGIBLE
- Examples:
  * "6.1020 or 6.1910" → need ONE
  * Student has [6.1020] → ELIGIBLE
  * Student has [6.1910] → ELIGIBLE

MIXED RULE:
- "A and (B or C)" = need A AND at least one of B or C
- "A or (B and C)" = need A OR need both B and C

ELIGIBILITY CHECK (do for EVERY course in context):
Step 1 - Read prerequisites EXACTLY from catalog context
Step 2 - Identify AND / OR / MIXED
Step 3 - AND  → ALL in {state.get('completed_courses')}? Any missing → NOT ELIGIBLE
Step 4 - OR   → ONE in {state.get('completed_courses')}? One present → ELIGIBLE
Step 5 - MIXED → check each part separately
Step 6 - Eligible → Answer/Plan | Not eligible → Not Eligible Courses
Step 7 - Unclear → Not in catalog

DECISION OUTPUT RULE — CRITICAL:
- Write ONLY the decision word after "DECISION:"
- Do NOT write all three options
- Correct:   DECISION: Eligible
- Correct:   DECISION: Not Eligible
- Correct:   DECISION: Need More Info
- WRONG:     DECISION: Eligible / Not Eligible / Need More Info

Output EXACTLY in this format:

Answer / Plan:
<only courses student IS eligible for with citation>

Not Eligible Courses:
- <course>: Missing prerequisites → <exactly which ones missing>

Why (requirements/prereqs satisfied):
<per course AND/OR logic shown explicitly>

Evidence:
- <catalog citations only>

Citations:
- <filename | chunk_id> for each course

Clarifying Questions:
- None

Assumptions / Not in catalog:
- <anything not in catalog context>

DECISION: <only one: Eligible OR Not Eligible OR Need More Info>

RISKS/ASSUMPTIONS:
- <risks>

NEXT STEP:
- <what student should do>
"""

    response = llm.invoke(prompt)
    text = response.content

    return {
        **state,
        "answer":                text,
        "decision":              _extract_section(text, "DECISION:"),
        "risks_assumptions":     _extract_section(text, "RISKS/ASSUMPTIONS:").split("\n"),
        "next_step":             _extract_section(text, "NEXT STEP:"),
        "why":                   _extract_section(text, "Why (requirements/prereqs satisfied):"),
        "citations":             _extract_section(text, "Citations:").split("\n"),
        "evidence":              _extract_section(text, "Evidence:").split("\n"),
        "assumptions":           _extract_section(text, "Assumptions / Not in catalog:").split("\n"),
        "is_not_in_catalog":     "Not in catalog" in text,
        "suggested_next_action": _extract_section(text, "NEXT STEP:"),
    }

# ══════════════════════════════════════════════════════
# AGENT 4 — VERIFIER NODE
# ══════════════════════════════════════════════════════
def verifier_node(state: AgentState) -> AgentState:

    valid_citations = [
        f"{c['filename']} | chunk_{c['chunk_id']}"
        for c in state['retrieved_chunks']
    ]

    prompt = f"""
You are a strict academic policy verifier/auditor.

Verify this course plan:
{state['answer']}

Valid catalog citations:
{valid_citations}

Student profile (INPUT data — does NOT need citations):
- Completed Courses: {state.get('completed_courses')}
- Grades: {state.get('grades')}

VERIFICATION RULES:
1. Catalog facts MUST have citations from valid list above
2. Student profile data does NOT need citations
3. AND prerequisites — ALL must be present, any missing → Not Eligible
4. OR prerequisites — ONE is enough → Eligible
5. DECISION must be exactly one of: Eligible / Not Eligible / Need More Info

PASS CONDITIONS (ALL must be true):
- Every catalog fact has valid citation
- AND/OR logic applied correctly
- DECISION is a single clean word — not all three options listed
- No catalog facts without citations

Output EXACTLY in this format:

VERIFICATION: PASSED / FAILED

UNSUPPORTED CLAIMS:
- <only catalog claims without valid citations>

CITATION COVERAGE:
- <cited> / <total catalog claims>

FEEDBACK:
- <specific fixes if FAILED>
"""

    response = llm.invoke(prompt)
    text = response.content

    retrieved_count = len(state['retrieved_chunks'])
    cited_count = sum(
        1 for c in valid_citations
        if c in state['answer']
    )
    coverage = round(cited_count / retrieved_count, 2) if retrieved_count > 0 else 0.0
    verification_passed = "VERIFICATION: PASSED" in text

    unsupported = _extract_section(text, "UNSUPPORTED CLAIMS:")
    unsupported_list = [
        u.strip("- ").strip()
        for u in unsupported.split("\n")
        if u.strip()
    ]

    retry_count = state.get('retry_count', 0) + 1

    return {
        **state,
        "verification_passed": verification_passed,
        "unsupported_claims":  unsupported_list,
        "citation_coverage":   coverage,
        "retry_count":         retry_count,
        "answer": state['answer'] if verification_passed else (
            state['answer'] + f"\n\n⚠️ VERIFICATION FAILED:\n{text}"
        )
    }

# ══════════════════════════════════════════════════════
# CONDITIONAL EDGES
# ══════════════════════════════════════════════════════
def should_clarify(state: AgentState) -> str:
    if state['needs_clarification']:
        return "end"
    return "retriever"

def should_rewrite(state: AgentState) -> str:
    if not state['verification_passed'] and state.get('retry_count', 0) < 2:
        return "planner"
    return "end"

# ══════════════════════════════════════════════════════
# BUILD GRAPH
# ══════════════════════════════════════════════════════
workflow = StateGraph(AgentState)

workflow.add_node("intake",    intake_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("planner",   planner_node)
workflow.add_node("verifier",  verifier_node)

workflow.set_entry_point("intake")

workflow.add_conditional_edges(
    "intake",
    should_clarify,
    {
        "end":       END,
        "retriever": "retriever"
    }
)

workflow.add_edge("retriever", "planner")
workflow.add_edge("planner",   "verifier")

workflow.add_conditional_edges(
    "verifier",
    should_rewrite,
    {
        "planner": "planner",
        "end":     END
    }
)

app = workflow.compile()

# ══════════════════════════════════════════════════════
# TEST RUN
# ══════════════════════════════════════════════════════
for step in app.stream({
    "student_query":        "What courses can I take next semester?",
    "completed_courses":    ["6.1020"],
    "grades":               {"6.1020": "A"},
    "target_program":       "Computer Science",
    "target_term":          "Fall",
    "max_credits":          48,
    "catalog_year":         "2024",
    "transfer_credits":     [],
    "missing_fields":       [],
    "clarifying_questions": [],
    "needs_clarification":  False,
    "retrieved_chunks":     [],
    "suggested_courses":    [],
    "risks_assumptions":    [],
    "decision":             "",
    "evidence":             [],
    "next_step":            "",
    "verification_passed":  False,
    "unsupported_claims":   [],
    "citation_coverage":    0.0,
    "retry_count":          0,
    "is_not_in_catalog":    False,
    "suggested_next_action":"",
    "answer":               "",
    "why":                  "",
    "citations":            [],
    "assumptions":          []
}):
    print("─── Node Output ───")
    for node, output in step.items():
        print(f"\n NODE: {node}")
        if 'answer' in output and output['answer']:
            print(output['answer'])
        elif 'clarifying_questions' in output and output['clarifying_questions']:
            print(output['clarifying_questions'])
        elif 'retrieved_chunks' in output:
            print(f"Retrieved {len(output['retrieved_chunks'])} chunks")
    print()

─── Node Output ───

 NODE: intake
Retrieved 0 chunks


DEBUG Retriever: Found 8 chunks
  → 6.1100 Computer Language Engineering.txt | score: 0.7643
  → 6.1040 Software Design.txt | score: 0.738
  → 6.5120 Formal Reasoning About Programs.txt | score: 0.7333
  → 6.1060 Software Performance Engineering.txt | score: 0.7308
  → 6.1120 Dynamic Computer Language Engineering.txt | score: 0.7293
  → cs_engineering_program.txt | score: 0.6143
  → ai_decision_program.txt | score: 0.0
  → cs_engineering_program.txt | score: 0.0
─── Node Output ───

 NODE: retriever
Retrieved 8 chunks

─── Node Output ───

 NODE: planner
Answer / Plan:
- 6.1040 Software Design: Eligible [6.1040.txt | chunk_0]
- 6.5120 Formal Reasoning About Programs: Eligible [6.5120.txt | chunk_0]
- 6.1120 Dynamic Computer Language Engineering: Eligible [6.1120.txt | chunk_0]

Not Eligible Courses:
- 6.1100 Computer Language Engineering: Missing prerequisites → 6.1910 [6.1100.txt | chunk_0]
- 6.1060 Software Performance Engineeri

In [61]:
# ══════════════════════════════════════════════════════
# TEST RUN 2: Complex Scenario
# ══════════════════════════════════════════════════════
for step in app.stream({
    "student_query":        "Which courses can I enroll in next Spring?",
    "completed_courses":    ["6.1020", "6.1010", "6.1200[J]"],
    "grades":               {"6.1020": "A", "6.1010": "B"},
    "target_program":       "Computer Science",
    "target_term":          "Spring",
    "max_credits":          24,
    "catalog_year":         "2024",
    "transfer_credits":     ["6.100A", "6.100B"],
    "missing_fields":       [],
    "clarifying_questions": [],
    "needs_clarification":  False,
    "retrieved_chunks":     [],
    "suggested_courses":    [],
    "risks_assumptions":    [],
    "decision":             "",
    "evidence":             [],
    "next_step":            "",
    "verification_passed":  False,
    "unsupported_claims":   [],
    "citation_coverage":    0.0,
    "retry_count":          0,
    "is_not_in_catalog":    False,
    "suggested_next_action":"",
    "answer":               "",
    "why":                  "",
    "citations":            [],
    "assumptions":          []
}):
    print("─── Node Output ───")
    for node, output in step.items():
        print(f"\n NODE: {node}")
        if 'answer' in output and output['answer']:
            print(output['answer'])
        elif 'clarifying_questions' in output and output['clarifying_questions']:
            print(output['clarifying_questions'])
        elif 'retrieved_chunks' in output:
            print(f"Retrieved {len(output['retrieved_chunks'])} chunks")
    print()

─── Node Output ───

 NODE: intake
Retrieved 0 chunks


DEBUG Retriever: Found 11 chunks
  → 6.1100 Computer Language Engineering.txt | score: 0.7643
  → 6.1040 Software Design.txt | score: 0.738
  → 6.5151 Large-scale Symbolic Systems.txt | score: 0.7371
  → 6.5120 Formal Reasoning About Programs.txt | score: 0.7333
  → 6.5130 Introduction to Program Synthesis (New).txt | score: 0.7316
  → 6.1060 Software Performance Engineering.txt | score: 0.7308
  → 6.5150 Large-scale Symbolic Systems.txt | score: 0.7307
  → 6.1120 Dynamic Computer Language Engineering.txt | score: 0.7293
  → cs_engineering_program.txt | score: 0.6143
  → ai_decision_program.txt | score: 0.0
  → cs_engineering_program.txt | score: 0.0
─── Node Output ───

 NODE: retriever
Retrieved 11 chunks

─── Node Output ───

 NODE: planner
Answer / Plan:
- 6.1100 Computer Language Engineering: Eligible
  - Prerequisites: 6.1020 and 6.1910
  - Student has: ['6.1020', '6.1010', '6.1200[J]']
  - Satisfies prerequisites: 6.1020 is